In [ ]:
%config InlineBackend.figure_formats = ['svg']
%matplotlib inline

# basic algebra and system libraries
import matplotlib.pyplot as plt
import numpy as np
import os
import os.path as path
import pandas as pd
import gc
from tqdm.notebook import tqdm
import datetime
import seaborn as sns
import random

# signal discrete fourier transform and loading
import scipy.signal as signal
import scipy.io.wavfile as wavfile
import scipy.fft as fft

# audio preprocessing, cross-validation, and others
from sklearn.base import TransformerMixin, BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight, resample
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, precision_score, recall_score


# construction of convolutional neural network
import keras
import keras.optimizers as optimizers
import keras.layers as layers
import keras.losses as losses
import keras.metrics as metrics
import keras.regularizers as regularizers
from tensorflow.keras import Input
from tensorflow.keras.callbacks import TensorBoard

# audio analysis
import librosa
import librosa.display

# processing speedup using GPU
import tensorflow as tf

In [ ]:
tf.config.set_visible_devices([], 'GPU')
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# Load audio data paths and metadata
metadata = []
data_dir = os.path.join("..", "data", "raw")
class_1_speakers = ['f1', 'f7', 'f8', 'm3', 'm6', 'm8']

for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.endswith(".wav"):
            speaker_id = file.split('_')[0]
            label = 1 if speaker_id in class_1_speakers else 0
            metadata.append({
                'path': os.path.join(root, file),
                'label': label
            })

metadata_df = pd.DataFrame(metadata)
metadata_df.head()
len(metadata)

In [ ]:
def load_and_split_audio_with_augmentation(file_path, target_sr=16000, segment_length=1, augment=False, label=None, augment_for_label=1):
    audio, sr = librosa.load(file_path, sr=target_sr)
    total_length = len(audio) / sr
    segments = []

    # Split audio into segments of length 'segment_length'
    for start in np.arange(0, total_length, segment_length):
        end = start + segment_length
        segment = audio[int(start * sr):int(end * sr)]

        # Pad if the segment is shorter
        if len(segment) < segment_length * sr:
            segment = np.pad(segment, (0, int(segment_length * sr) - len(segment)), mode='constant')

        # Apply augmentation only if augment=True and the label matches augment_for_label
        if augment and label == augment_for_label:
            segment = augment_audio(segment, sr=sr)

        segments.append(segment)

    return segments

def generate_spectrogram_batch(segments, sr=16000):
    spectrograms = []
    for segment in segments:
        # Generate Mel Spectrogram
        S = librosa.feature.melspectrogram(y=segment, sr=sr, n_fft=2048, hop_length=512, n_mels=128)
        S_DB = librosa.power_to_db(S, ref=np.max)
        spectrograms.append(S_DB)
    return spectrograms

# Apply data augmentation
def augment_audio(audio, sr=16000):
    augmented = False

    while not augmented:
        # Time Shifting
        if random.random() < 0.5:
            shift = int(random.uniform(-0.1, 0.1) * sr)  # Shift by ±0.1 seconds
            audio = np.roll(audio, shift)
            augmented = True

        # Pitch Shifting
        if random.random() < 0.5:
            pitch_shift = random.uniform(-2, 2)  # Shift pitch by ±2 semitones
            audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=pitch_shift)
            augmented = True

        # Time Stretching
        if random.random() < 0.5:
            stretch_factor = random.uniform(0.8, 1.2)  # Stretch by 80%-120%
            audio = librosa.effects.time_stretch(np.asfortranarray(audio), rate=stretch_factor)
            # Ensure the audio length matches the original
            if len(audio) > sr:
                audio = audio[:sr]
            else:
                audio = np.pad(audio, (0, sr - len(audio)), mode='constant')
            augmented = True

    # Add Noise
    noise_amp = 0.005 * np.random.uniform() * np.amax(audio)
    audio = audio + noise_amp * np.random.normal(size=audio.shape)
    augmented = True

    return audio

In [ ]:
# Specify which labels to augment
labels_to_augment = [1]

# Apply audio splitting and augmentation
metadata_df['segments'] = [
    load_and_split_audio_with_augmentation(
        file_path=x, 
        augment=True if label in labels_to_augment else False,
        label=label,
        augment_for_label=1
    )
    for x, label in tqdm(zip(metadata_df['path'], metadata_df['label']), desc="Loading and Splitting with Augmentation")
]

# Explode the DataFrame to handle individual segments
metadata_df = metadata_df.explode('segments', ignore_index=True)

# Generate spectrograms for all segments
metadata_df['spectrograms'] = [
    generate_spectrogram_batch([segment])[0]
    for segment in tqdm(metadata_df['segments'], desc="Generating Spectrograms")
]

In [ ]:
# Plot some example spectrograms
plt.figure(figsize=(10, 5))
for i in range(3):
    plt.subplot(1, 3, i+1)
    librosa.display.specshow(metadata_df.iloc[i]['spectrograms'], sr=16000, hop_length=512, x_axis='time', y_axis='mel')
    plt.title(f"Label: {metadata_df.iloc[i]['label']}")
    plt.colorbar(format='%+2.0f dB')
plt.tight_layout()
plt.show()

# Histogram of spectrogram values per class
class_0_spectrograms = metadata_df[metadata_df['label'] == 0]['spectrograms'].values
class_1_spectrograms = metadata_df[metadata_df['label'] == 1]['spectrograms'].values

# Flatten spectrogram values
class_0_values = np.concatenate([s.ravel() for s in class_0_spectrograms])
class_1_values = np.concatenate([s.ravel() for s in class_1_spectrograms])

# Plot histograms
plt.figure(figsize=(12, 6))
plt.hist(class_0_values, bins=50, alpha=0.5, label='Class 0', color='blue')
plt.hist(class_1_values, bins=50, alpha=0.5, label='Class 1', color='orange')
plt.title("Histogram of Spectrogram Values per Class")
plt.xlabel("Value (dB)")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
# Train-test split before augmentation and audio splitting
train_df, test_df = train_test_split(metadata_df, test_size=0.2, random_state=42, stratify=metadata_df['label'])

# Process training set with augmentation
train_df['segments'] = [
    load_and_split_audio_with_augmentation(
        file_path=x,
        target_sr=16000,
        segment_length=1,
        augment=True,  # Enable augmentation for training data
        label=label,
        augment_for_label=1
    ) for x, label in tqdm(zip(train_df['path'], train_df['label']), desc="Processing Train Audio")
]

# Process test set without augmentation
test_df['segments'] = [
    load_and_split_audio_with_augmentation(
        file_path=x,
        target_sr=16000,
        segment_length=1,
        augment=False,  # No augmentation for test data
        label=label
    ) for x, label in tqdm(zip(test_df['path'], test_df['label']), desc="Processing Test Audio")
]

# Explode the segments for train and test
train_df = train_df.explode('segments')
test_df = test_df.explode('segments')

# Generate spectrograms
train_df['spectrograms'] = generate_spectrogram_batch(train_df['segments'].tolist())
test_df['spectrograms'] = generate_spectrogram_batch(test_df['segments'].tolist())

# Apply oversampling to balance classes in the training data
# Separate majority and minority classes
train_majority = train_df[train_df['label'] == 0]
train_minority = train_df[train_df['label'] == 1]

# Oversample the minority class
train_minority_oversampled = resample(
    train_minority,
    replace=True,
    n_samples=len(train_majority),
    random_state=42
)

# Combine the majority class with the oversampled minority class
train_df_balanced = pd.concat([train_majority, train_minority_oversampled])

# Prepare features and labels from the balanced dataset
X_train = np.stack(train_df_balanced['spectrograms'].values)[..., np.newaxis]
Y_train = train_df_balanced['label'].values

# Prepare the test data
X_test = np.stack(test_df['spectrograms'].values)[..., np.newaxis]
Y_test = test_df['label'].values

In [ ]:
# Check class distribution before augmentation and splitting
print("Class distribution before augmentation and splitting:")
print(train_df['label'].value_counts())

# Check class distribution after oversampling
print("\nClass distribution after augmentation and oversampling:")
print(train_df_balanced['label'].value_counts())

# Plot before augmentation and splitting
plt.figure(figsize=(10, 5))
plt.subplot(1, 3, 1)
train_df['label'].value_counts().plot(kind='bar')
plt.title("Before Augmentation")
plt.xlabel("Class")
plt.ylabel("Count")

# Plot after oversampling
plt.subplot(1, 3, 3)
train_df_balanced['label'].value_counts().plot(kind='bar')
plt.title("After Augmentation and Oversampling")
plt.xlabel("Class")
plt.ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Create basic CNN model
def create_cnn():
    inputs = Input(shape=(128, 32, 1))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=optimizers.Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = create_cnn()

# Set up TensorBoard callback
log_dir = os.path.join("logs", "fit") + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

# Train model
history = model.fit(
    X_train, Y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, Y_test),
    callbacks=[tensorboard_callback],
    class_weight=dict(enumerate(compute_class_weight('balanced', classes=np.unique(Y_train), y=Y_train)))
)

In [ ]:
# Plot training history
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.show()

### Estimate Classification Confidence using Monte Carlo Dropout

In [ ]:
# Add Dropout to the CNN model
def create_cnn_with_dropout():
    inputs = Input(shape=(128, 32, 1))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)  # Dropout
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.Dropout(0.3)(x)  # Dropout
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)  # Dropout
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-4), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    return model

# Force Monte Carlo Dropout
import functools

def enable_monte_carlo_dropout(model):
    for layer in model.layers:
        if isinstance(layer, layers.Dropout):
            layer.call = functools.partial(layer.call, training=True)

def mc_dropout_predict(model, X, n_iter=10):
    enable_monte_carlo_dropout(model)
    preds = []
    for _ in range(n_iter):
        preds.append(model.predict(X, verbose=0))
    return np.array(preds)

# Train model with Dropout
mc_model = create_cnn_with_dropout()
mc_model.fit(X_train, Y_train, epochs=5, validation_data=(X_test, Y_test), batch_size=32)

n_passes = 20
mc_preds = mc_dropout_predict(mc_model, X_test, n_iter=n_passes)
mean_preds = mc_preds.mean(axis=0)
std_preds = mc_preds.std(axis=0)

# Ensemble learning
def train_ensemble(n_models=3):
    ensemble = []
    for i in range(n_models):
        model_i = create_cnn_with_dropout()
        model_i.fit(X_train, Y_train, epochs=5, validation_data=(X_test, Y_test), batch_size=32, verbose=0)
        ensemble.append(model_i)
    return ensemble

ensemble = train_ensemble(n_models=3)

def ensemble_predict(ensemble, X):
    all_preds = [m.predict(X, verbose=0) for m in ensemble]
    return np.mean(all_preds, axis=0), np.std(all_preds, axis=0)

ensemble_mean, ensemble_std = ensemble_predict(ensemble, X_test)

# Compare MC Dropout and Ensemble predictions
# mean_preds, std_preds vs ensemble_mean, ensemble_std
print("MC Dropout prediction mean:", mean_preds[:5].ravel())
print("MC Dropout std deviation:", std_preds[:5].ravel())
print("Ensemble prediction mean:", ensemble_mean[:5].ravel())
print("Ensemble std deviation:", ensemble_std[:5].ravel())

In [ ]:
# Generate predictions and calculate metrics
Y_pred = (model.predict(X_test) > 0.5).astype('int32')

# Calculate accuracy
accuracy = accuracy_score(Y_test, Y_pred)

# Compute the macro-average F1 score
macro_f1 = f1_score(Y_test, Y_pred, average='macro')

# Calculate precision and recall
precision = precision_score(Y_test, Y_pred, average='macro')
recall = recall_score(Y_test, Y_pred, average='macro')

# Output the metrics
print(f'Accuracy: {accuracy:.4f}')
print(f'Macro F1: {macro_f1:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')

# Show class report
class_report = classification_report(Y_test, Y_pred)
print(class_report)

In [ ]:
speakers = [os.path.basename(metadata_df['path'].iloc[i]).split('_')[0] for i in range(len(X_test))]
Y_pred_df = pd.DataFrame(Y_pred, columns=['Predictions'])
Y_pred_df['True_Labels'] = Y_test

unique_speakers = set(speakers)
print(unique_speakers)

fig, axes = plt.subplots(len(unique_speakers), 1, figsize=(8, 6 * len(unique_speakers)), squeeze=False)

for i, speaker in enumerate(unique_speakers):
    speaker_indices = [j for j, spk in enumerate(speakers) if spk == speaker]

    if not speaker_indices:
        print(f"No samples found for speaker {speaker}. Skipping...")
        continue

    # Extract true labels and predictions for the current speaker using the indices
    Y_test_speaker = Y_pred_df['True_Labels'].iloc[speaker_indices].values
    Y_pred_speaker = Y_pred_df['Predictions'].iloc[speaker_indices].values

    # Compute confusion matrix
    conf_matrix = confusion_matrix(Y_test_speaker, Y_pred_speaker)

    # Plot confusion matrix
    ax = axes[i, 0]
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Class 0', 'Class 1'],
                yticklabels=['Class 0', 'Class 1'], ax=ax)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_title(f'Confusion Matrix for Speaker {speaker}')

plt.tight_layout()
plt.show()


In [ ]:
# Plot combined confusion matrix
conf_matrix = confusion_matrix(Y_test, Y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [19]:
# Extract missed examples
misclassified_indices = np.where(Y_test != Y_pred.flatten())[0]

In [ ]:
# Plot some misclassified spectrograms
num_to_plot = min(10, len(misclassified_indices))  # Adjust the number to plot as needed
plt.figure(figsize=(15, 10))

for i in range(num_to_plot):
    idx = misclassified_indices[i]
    spectrogram = X_test[idx].squeeze()  # Remove channel dimension

    plt.subplot(2, 5, i + 1)  # Plot in a 2x5 grid for 10 samples
    librosa.display.specshow(spectrogram, sr=16000, hop_length=512, x_axis='time', y_axis='mel')
    plt.title(f"True: {Y_test[idx]}, Predicted: {Y_pred[idx][0]}")
    plt.colorbar(format="%+2.0f dB")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.manifold import TSNE
import seaborn as sns

# Extract the penultimate layer activations
intermediate_model = keras.Model(inputs=model.input, outputs=model.layers[-2].output)
latent_features = intermediate_model.predict(X_test)

# Reduce dimensionality to 2D using TSNE
tsne = TSNE(n_components=2, random_state=42)
latent_2d = tsne.fit_transform(latent_features)

# Plot TSNE results
plt.figure(figsize=(10, 8))
sns.scatterplot(x=latent_2d[:, 0], y=latent_2d[:, 1], hue=Y_test, palette='deep')
plt.title('t-SNE visualization of Latent Features')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.show()

In [ ]:
from sklearn.decomposition import PCA

# Step 1: Extract the penultimate layer activations (latent features)
intermediate_model = keras.Model(inputs=model.input, outputs=model.layers[-2].output)
latent_features = intermediate_model.predict(X_test)

# Step 2: Apply PCA to reduce the latent features to 2 dimensions
pca = PCA(n_components=2)
latent_2d_pca = pca.fit_transform(latent_features)

# Step 3: Plot the PCA results
plt.figure(figsize=(10, 8))
sns.scatterplot(x=latent_2d_pca[:, 0], y=latent_2d_pca[:, 1], hue=Y_test, palette='deep', alpha=0.6)
plt.title('PCA Visualization of Latent Features')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

pca_3d = PCA(n_components=3)
latent_3d_pca = pca_3d.fit_transform(latent_features)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(latent_3d_pca[:, 0], latent_3d_pca[:, 1], latent_3d_pca[:, 2], c=Y_test, cmap='viridis', alpha=0.6)
ax.set_title('3D PCA Visualization of Latent Features')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
plt.colorbar(scatter)
plt.show()